In [ ]:
import pandas as pd
import numpy as np
import re
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

### 1. Assigning word indicies

In [ ]:
data = pd.read_csv("train.En.csv")
data = data.dropna()

In [ ]:
index = 1
vocab = {}
for tweet in data['tweet']:
    tweet = str(tweet)

    # Clean text
    tweet = tweet.lower()
    tweet = re.sub(r'[^a-zA-Z0-9\s]', '', tweet)
    tweet = tweet.strip()

    for word in tweet.split(" "):
        if word not in vocab:
            vocab[word] = index
            index += 1
        else:
            continue

In [ ]:
def encode_tweet(tweet, vocab, max_len):
  if type(tweet) != str:
    tweet = str(tweet)
  tokens = tweet.lower().split()
  ids = [vocab.get(token, 1) for token in tokens]

  ids = ids[:max_len]

  padding_len = max_len - len(ids)
  ids = ids + ([0] * padding_len)

  return torch.tensor(ids)

In [ ]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        # Return the encoded text and its label (0-4)
        x = encode_tweet(self.texts[idx], self.vocab, self.max_len)
        y = torch.tensor(self.labels[idx])
        return x, y

In [ ]:
tweets = data['tweet'].tolist()
labels = []
for i in range(len(data['sarcasm'])):
  # For some reason i need the labels to be backwards for this model to work on the training set... idk
  if data['sarcasm'].iloc[i] == 1.0: labels.append(0)
  # just doing sarcasm for now
  #elif data['irony'].iloc[i] == 1.0: labels.append(2)
  #elif data['satire'].iloc[i] == 1.0: labels.append(3)
  #elif data['understatement'].iloc[i] == 1.0: labels.append(4)
  #elif data['overstatement'].iloc[i] == 1.0: labels.append(5)
  #elif data['rhetorical_question'].iloc[i] == 1.0: labels.append(6)
  else: labels.append(1)

dataset = TextDataset(tweets, labels, vocab, max_len=30)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

In [ ]:
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_layers, num_classes):
        super(TransformerClassifier, self).__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Embedding(100, d_model) # max length should be < 100

        # encoder
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # classifier
        self.fc = nn.Linear(d_model, num_classes)

    def forward(self, x):
        positions = torch.arange(0, x.size(1)).expand(x.size(0), x.size(1)).to(x.device)

        x = self.embedding(x) + self.pos_embedding(positions)
        x = self.transformer_encoder(x)
        x = x.mean(dim=1)

        logits = self.fc(x)
        return logits

In [ ]:
model = TransformerClassifier(vocab_size=len(vocab), d_model=128, nhead=8, num_layers=2, num_classes=2)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
num_epochs = 5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for texts, labels in dataloader:
        texts, labels = texts.to(device), labels.to(device)

        # clear gradient
        optimizer.zero_grad()

        # forward pass
        outputs = model(texts)
        loss = criterion(outputs, labels)

        # backward pass
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(dataloader):.4f}")

Epoch [1/5], Loss: 0.0003
Epoch [2/5], Loss: 0.0003
Epoch [3/5], Loss: 0.0001
Epoch [4/5], Loss: 0.0001
Epoch [5/5], Loss: 0.0001


In [ ]:
def predict(text, model, vocab, max_len):
    model.eval()
    with torch.no_grad():
        input_tensor = encode_tweet(text, vocab, max_len).unsqueeze(0)
        logits = model(input_tensor.to(device))
        prediction = torch.argmax(logits, dim=1)
        return prediction.item()

In [ ]:
test_data = pd.read_csv("task_A_En_test.csv")

In [ ]:
correct = 0
for i in range(len(test_data)):
  y_hat = predict(test_data.at[i, 'text'], model, vocab, 30)
  if y_hat == test_data.at[i, 'sarcastic']:
    correct += 1

print(f"Error: {round(1 - (correct / len(test_data)), 2)}")

Error: 0.2
